In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)
import joblib

# Find the project root reliably
ROOT = Path.cwd()
if not (ROOT / "data" / "reviews.csv").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocessing import clean_texts, drop_trailing_fragment

DATA_PATH = ROOT / "data" / "reviews.csv"
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

print("Project root:", ROOT)
print("Dataset exists:", DATA_PATH.exists())
print("Model directory:", MODEL_DIR)

Project root: c:\Users\poona\OneDrive\Desktop\parkh
Dataset exists: True
Model directory: c:\Users\poona\OneDrive\Desktop\parkh\models


In [2]:
df = pd.read_csv(DATA_PATH)

print("Original shape:", df.shape)
print("Columns:", df.columns.tolist())

required_columns = {"text_", "label"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df = df.drop_duplicates().reset_index(drop=True)
df["text_"] = df["text_"].fillna("").astype(str).str.strip()
df["label"] = df["label"].astype(str).str.strip()

df = df[df["text_"] != ""].copy()

print("\nUsable shape:", df.shape)
print("\nLabel counts:")
print(df["label"].value_counts())
print("\nMissing values:")
print(df[["text_", "label"]].isna().sum())

Original shape: (40432, 4)
Columns: ['category', 'rating', 'label', 'text_']

Usable shape: (40420, 4)

Label counts:
label
OR    20215
CG    20205
Name: count, dtype: int64

Missing values:
text_    0
label    0
dtype: int64


In [3]:
X = df["text_"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

print("\nTraining label distribution:")
print(y_train.value_counts())

print("\nTesting label distribution:")
print(y_test.value_counts())

Training reviews: 32336
Testing reviews: 8084

Training label distribution:
label
OR    16172
CG    16164
Name: count, dtype: int64

Testing label distribution:
label
OR    4043
CG    4041
Name: count, dtype: int64


In [4]:
authenticity_model = Pipeline([
    (
        "preprocessing",
        FunctionTransformer(clean_texts, validate=False)
    ),
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            max_features=20000,
            ngram_range=(1, 2)
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

authenticity_model.fit(X_train, y_train)

print("Model training completed successfully!")
print("Classes:", authenticity_model.named_steps["classifier"].classes_)

Model training completed successfully!
Classes: ['CG' 'OR']


In [5]:
# Predict labels for unseen test reviews
y_pred = authenticity_model.predict(X_test)

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")

print(f"Test Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Macro F1-score: {macro_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=["CG", "OR"],
        target_names=["Computer-generated", "Genuine/original"],
        digits=4
    )
)

print("Confusion Matrix (rows = actual, columns = predicted)")
print("Order: CG, OR")
print(confusion_matrix(y_test, y_pred, labels=["CG", "OR"]))

Test Accuracy: 0.9031 (90.31%)
Macro F1-score: 0.9031

Classification Report:
                    precision    recall  f1-score   support

Computer-generated     0.9062    0.8993    0.9027      4041
  Genuine/original     0.9001    0.9070    0.9035      4043

          accuracy                         0.9031      8084
         macro avg     0.9032    0.9031    0.9031      8084
      weighted avg     0.9032    0.9031    0.9031      8084

Confusion Matrix (rows = actual, columns = predicted)
Order: CG, OR
[[3634  407]
 [ 376 3667]]


In [6]:
sample_reviews = [
    "The product is excellent and works exactly as expected.",
    "I absolutely love this product. It is amazing, wonderful, perfect, and the best purchase ever!",
    "The item arrived yesterday. I have not used it yet.",
    "Very poor quality. It stopped working after two days."
]

sample_predictions = authenticity_model.predict(sample_reviews)
sample_probabilities = authenticity_model.predict_proba(sample_reviews)

classes = authenticity_model.named_steps["classifier"].classes_

for review, prediction, probabilities in zip(
    sample_reviews, sample_predictions, sample_probabilities
):
    probability_by_class = dict(zip(classes, probabilities))

    print("Review:", review)
    print("Predicted class:", prediction)
    print(f"CG probability: {probability_by_class['CG']:.2%}")
    print(f"OR probability: {probability_by_class['OR']:.2%}")
    print("-" * 60)

Review: The product is excellent and works exactly as expected.
Predicted class: OR
CG probability: 18.02%
OR probability: 81.98%
------------------------------------------------------------
Review: I absolutely love this product. It is amazing, wonderful, perfect, and the best purchase ever!
Predicted class: CG
CG probability: 51.66%
OR probability: 48.34%
------------------------------------------------------------
Review: The item arrived yesterday. I have not used it yet.
Predicted class: CG
CG probability: 54.11%
OR probability: 45.89%
------------------------------------------------------------
Review: Very poor quality. It stopped working after two days.
Predicted class: OR
CG probability: 23.35%
OR probability: 76.65%
------------------------------------------------------------


In [7]:
MODEL_PATH = MODEL_DIR / "authenticity_model.joblib"

joblib.dump(authenticity_model, MODEL_PATH)

print("Model saved to:", MODEL_PATH)
print("File exists:", MODEL_PATH.exists())
print(f"File size: {MODEL_PATH.stat().st_size / 1024:.2f} KB")

Model saved to: c:\Users\poona\OneDrive\Desktop\parkh\models\authenticity_model.joblib
File exists: True
File size: 908.97 KB


In [8]:
loaded_model = joblib.load(MODEL_PATH)

test_predictions = loaded_model.predict(X_test)
loaded_accuracy = accuracy_score(y_test, test_predictions)

print(f"Original model accuracy: {accuracy:.4f}")
print(f"Loaded model accuracy:   {loaded_accuracy:.4f}")

assert np.array_equal(y_pred, test_predictions), (
    "Predictions differ after reloading the model!"
)

print("SUCCESS: Saved model loads correctly and predictions match.")

Original model accuracy: 0.9031
Loaded model accuracy:   0.9031
SUCCESS: Saved model loads correctly and predictions match.


In [11]:
def explain_prediction(review, top_n=8):
    # Apply the same preprocessing and vectorization used by the model
    processed_text = clean_texts([review])
    vectorizer = authenticity_model.named_steps["tfidf"]
    classifier = authenticity_model.named_steps["classifier"]

    features = vectorizer.transform(processed_text)
    predicted_class = authenticity_model.predict([review])[0]
    probabilities = authenticity_model.predict_proba([review])[0]
    probability_by_class = dict(zip(classifier.classes_, probabilities))

    # Logistic Regression coefficients indicate direction of association
    coefficients = classifier.coef_[0]
    feature_indices = features.indices
    feature_values = features.data

    contributions = []
    for index, value in zip(feature_indices, feature_values):
        contributions.append(
            (vectorizer.get_feature_names_out()[index],
             float(value * coefficients[index]))
        )

    contributions.sort(key=lambda item: abs(item[1]), reverse=True)

    print("Review:", review)
    print("Predicted class:", predicted_class)
    print(f"CG probability: {probability_by_class['CG']:.2%}")
    print(f"OR probability: {probability_by_class['OR']:.2%}")
    print("\nMost influential present features:")
    for word, contribution in contributions[:top_n]:
        direction = "toward OR" if contribution > 0 else "toward CG"
        print(f"  {word!r}: {direction} ({contribution:+.4f})")

    return predicted_class, probability_by_class, contributions[:top_n]

In [12]:
explain_prediction(
    "The product is excellent and works exactly as expected."
)

Review: The product is excellent and works exactly as expected.
Predicted class: OR
CG probability: 18.02%
OR probability: 81.98%

Most influential present features:
  'as': toward OR (+0.4959)
  'and': toward OR (+0.3143)
  'exactly': toward OR (+0.2937)
  'product is': toward OR (+0.2248)
  'and works': toward OR (+0.2223)
  'the product': toward OR (+0.2064)
  'excellent and': toward OR (+0.1933)
  'exactly as': toward OR (+0.1809)


('OR',
 {'CG': np.float64(0.18023692060892826), 'OR': np.float64(0.8197630793910717)},
 [('as', 0.49592256540998314),
  ('and', 0.3142657242770961),
  ('exactly', 0.2937171501605666),
  ('product is', 0.22481250547997722),
  ('and works', 0.22232030045987589),
  ('the product', 0.2063812367484454),
  ('excellent and', 0.19332585251891957),
  ('exactly as', 0.18086753578410458)])